# Alzheimer Detection — Exploratory Data Analysis

This notebook walks through:
1. Dataset inspection
2. Visualising sample MRI images per class
3. Feature extraction and dimensionality exploration
4. Quick model training and evaluation


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from utils import DATA_RAW, CLASSES, CLASS_TO_IDX
from preprocess import load_image, extract_features, build_dataset

sns.set_style('whitegrid')
print('Imports OK')

## 1. Dataset Size per Class

In [ ]:
for split in ('train', 'test'):
    print(f'\n{split.upper()}')
    for cls in CLASSES:
        folder = DATA_RAW / split / cls
        n = len(list(folder.glob('*.jpg'))) + len(list(folder.glob('*.png'))) if folder.exists() else 0
        print(f'  {cls:<25} {n:>5} images')

## 2. Sample Images per Class

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(14, 11))
for row, cls in enumerate(CLASSES):
    folder = DATA_RAW / 'train' / cls
    images = sorted(folder.glob('*.jpg'))[:5] if folder.exists() else []
    for col in range(5):
        ax = axes[row][col]
        if col < len(images):
            img = load_image(images[col])
            ax.imshow(img, cmap='bone')
        ax.axis('off')
        if col == 0:
            ax.set_title(cls, fontsize=9, loc='left', pad=4)
plt.suptitle('Sample MRI Scans per Class', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Mean Brain Image per Class

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, cls in zip(axes, CLASSES):
    folder = DATA_RAW / 'train' / cls
    if not folder.exists():
        ax.set_title(f'{cls}\n(missing)')
        ax.axis('off')
        continue
    imgs = [load_image(p).astype(np.float32) for p in list(folder.glob('*.jpg'))[:100]]
    mean_img = np.mean(imgs, axis=0)
    ax.imshow(mean_img, cmap='bone')
    ax.set_title(cls, fontsize=10)
    ax.axis('off')
plt.suptitle('Mean Brain Image per Class (first 100 samples)', fontsize=12)
plt.tight_layout()
plt.show()

## 4. Feature Extraction

In [ ]:
X_train, y_train = build_dataset('train')
X_test,  y_test  = build_dataset('test')
print(f'Train features: {X_train.shape}')
print(f'Test  features: {X_test.shape}')
print(f'Feature vector length: {X_train.shape[1]}')

## 5. PCA Visualisation

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)

palette = ['#22c55e', '#eab308', '#f97316', '#ef4444']
fig, ax = plt.subplots(figsize=(8, 6))
for i, cls in enumerate(CLASSES):
    mask = y_train == i
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               label=cls, alpha=0.5, s=20, color=palette[i])
ax.legend(fontsize=9)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA of Feature Space')
plt.tight_layout()
plt.show()

## 6. Quick Train & Evaluate

In [ ]:
from train import train
results = train(model_key='rf', cross_validate=False)